<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/numpy/ImpliedWellandGOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# The generate_arps_profile function is not used for the new well production calculation.

# Define the start date as a variable
START_DATE_STR = '2023-12-01' # This represents the M0 month, e.g., 'YYYY-MM-DD'
timeline_length = 24 # Number of months to forecast

GAS_REGIONS = ['Appalachia', 'Haynesville']

# 1. Load the CSV files
df = pd.read_csv("CurveData.csv")
factor_df = pd.read_csv("FactorDataT.csv")


In [40]:
# Load the target primary production data
pPR_df = pd.read_csv("pPR.csv")

# Load the target non-primary production data
npPR_df = pd.read_csv("npPR.csv")

# Dictionary to store the calculated well count vectors for each region
calculated_well_counts_neg_allowed = {}

# Dictionary to store the calculated GOR ratio vectors for each region
calculated_gor_ratios = {}

In [41]:
# Identify our timeline length in months for the new decline curve (M1 through M24)
# Corrected to use uppercase 'M' for column names to match the CSV data
month_cols = [f'M{i+1}' for i in range(timeline_length)]
num_months = len(month_cols)

In [42]:
def generate_base_schedule_matrix(type_curve, num_months, ip_improvement_vector):
    """
    Generates a reusable base schedule matrix where each row represents a well vintage
    (the month a batch of wells is turned online) and each column represents a calendar timeline month.

    Parameters:
    type_curve (np.ndarray): The *normalized* production profile vector for a single standard well.
    num_months (int): Total number of months to model in the forecast timeline.
    ip_improvement_vector (np.ndarray): A vector of IP rates for each start_month (vintage).

    Returns:
    np.ndarray: A 2D matrix of shape (num_months, num_months) filled with shifted type curves,
                scaled by the appropriate IP rate for each vintage.
    """
    base_matrix = np.zeros((num_months, num_months))

    for start_month in range(num_months):
        # Calculate how many months are left in the forecast timeline from this start month forward
        months_remaining = num_months - start_month

        # Determine how much of the type curve fits into the remaining timeline window
        curve_length = min(len(type_curve), months_remaining)

        # Populate the row starting from the diagonal (the month the well goes online)
        # Scale the normalized type curve by the IP rate for that specific vintage
        base_matrix[start_month, start_month:start_month + curve_length] = type_curve[:curve_length] * ip_improvement_vector[start_month]

    return base_matrix

In [43]:
# Iterate through each region to perform the inverse calculation
for region, group in df.groupby('Region'):
    # --- Re-calculate necessary components for the region (as done in the forward pass) ---

    # Extract the new well decline curve (normalized)
    normalized_new_well_decline_curve_values_original = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    if not new_well_decline_curve_row.empty:
        normalized_new_well_decline_curve_values_original = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Retrieve P1 and P1end for the current region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]

    p1_end_rate = 0.0
    p1_end_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1end']
    if not p1_end_rate_row.empty:
        p1_end_rate = p1_end_rate_row.values[0]

    # Generate IP Improvement Vector
    ip_improvement_vector = np.linspace(ip_rate, p1_end_rate, num_months)

    # Generate the base production matrix (which includes IP improvement per vintage)
    base_prod_matrix = generate_base_schedule_matrix(normalized_new_well_decline_curve_values_original, num_months, ip_improvement_vector)

    # Calculate legacy production values (M1 to M24)
    m0_value = 0.0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0'] * 1000
    if not m0_series_row.empty:
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    legacy_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values * m0_value

    # --- Get target total primary production from pPR_df for M1 to M24 ---
    # Check if the region column exists in pPR_df
    if region not in pPR_df.columns:
        print(f"Warning: No target production column found for region {region} in pPR.csv. Skipping.")
        calculated_well_counts_neg_allowed[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in pPR.csv
    # Multiply by 1000 to convert from '000 units to full units, matching the legacy production scale
    target_production_values_M1_to_M24 = pd.to_numeric(pPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values * 1000

    # Calculate the desired new well production profile (M1 to M24)
    desired_new_well_production = target_production_values_M1_to_M24 - legacy_decline_curve_values

    # Solve the linear system: A @ W = P_new_total_desired
    # Where A_matrix is the transpose of base_prod_matrix (base_prod_matrix[i, k] is the contribution
    # of well vintage 'i' to calendar month 'k' production, already scaled by IP improvement).
    A_matrix = base_prod_matrix.T

    try:
        # The result 'well_count_vector_calculated' represents the number of new wells
        # starting in each month (M1 to M24) that are needed.
        well_count_vector_calculated = np.linalg.solve(A_matrix, desired_new_well_production)
        # Allowing negative values, and rounding to nearest integer
        calculated_well_counts_neg_allowed[region] = well_count_vector_calculated.round(0).astype(int)
    except np.linalg.LinAlgError as e:
        print(f"Could not solve for well counts for region {region}: {e}. This might happen if the production matrix is singular or ill-conditioned. Skipping this region.")
        calculated_well_counts_neg_allowed[region] = np.full(num_months, np.nan) # Fill with NaN if cannot solve


In [44]:
# Iterate through each region to perform the inverse calculation
for region, group in df.groupby('Region'):
    # --- Re-calculate necessary components for the region (as done in the forward pass) ---

    # Extract the new well decline curve (normalized)
    normalized_new_well_decline_curve_values_original = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    if not new_well_decline_curve_row.empty:
        normalized_new_well_decline_curve_values_original = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Retrieve P1 and P1end for the current region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]

    p1_end_rate = 0.0
    p1_end_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1end']
    if not p1_end_rate_row.empty:
        p1_end_rate = p1_end_rate_row.values[0]

    # Generate IP Improvement Vector
    ip_improvement_vector = np.linspace(ip_rate, p1_end_rate, num_months)

    # Generate the base production matrix (which includes IP improvement per vintage)
    base_prod_matrix = generate_base_schedule_matrix(normalized_new_well_decline_curve_values_original, num_months, ip_improvement_vector)

    # Calculate legacy production values (M1 to M24)
    m0_value = 0.0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0'] * 1000
    if not m0_series_row.empty:
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    legacy_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values * m0_value

    # --- Get target total primary production from pPR_df for M1 to M24 ---
    # Check if the region column exists in pPR_df
    if region not in pPR_df.columns:
        print(f"Warning: No target production column found for region {region} in pPR.csv. Skipping.")
        calculated_well_counts_neg_allowed[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in pPR.csv
    # Multiply by 1000 to convert from '000 units to full units, matching the legacy production scale
    target_production_values_M1_to_M24 = pd.to_numeric(pPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values * 1000

    # Calculate the desired new well production profile (M1 to M24)
    desired_new_well_production = target_production_values_M1_to_M24 - legacy_decline_curve_values

    # Solve the linear system: A @ W = P_new_total_desired
    # Where A_matrix is the transpose of base_prod_matrix (base_prod_matrix[i, k] is the contribution
    # of well vintage 'i' to calendar month 'k' production, already scaled by IP improvement).
    A_matrix = base_prod_matrix.T

    try:
        # The result 'well_count_vector_calculated' represents the number of new wells
        # starting in each month (M1 to M24) that are needed.
        well_count_vector_calculated = np.linalg.solve(A_matrix, desired_new_well_production)
        # Allowing negative values, and rounding to nearest integer
        calculated_well_counts_neg_allowed[region] = well_count_vector_calculated.round(0).astype(int)
    except np.linalg.LinAlgError as e:
        print(f"Could not solve for well counts for region {region}: {e}. This might happen if the production matrix is singular or ill-conditioned. Skipping this region.")
        calculated_well_counts_neg_allowed[region] = np.full(num_months, np.nan) # Fill with NaN if cannot solve


In [45]:
# Create a DataFrame for the calculated well counts
calculated_well_counts_neg_allowed_df = pd.DataFrame(calculated_well_counts_neg_allowed, index=month_cols)
calculated_well_counts_neg_allowed_df['L48'] = calculated_well_counts_neg_allowed_df.sum(axis=1) # Sum across regions for L48 total

print("\n--- Calculated Well Count Vector by Vintage (M1 to M24) - Negative Counts Allowed ---")
display(calculated_well_counts_neg_allowed_df.style.format("{:,.0f}"))


--- Calculated Well Count Vector by Vintage (M1 to M24) - Negative Counts Allowed ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
M1,41,-556,0,41,-174,-644,"-1,292"
M2,112,"1,415",269,97,"1,580","1,879","5,352"
M3,-182,"-1,510",7,-31,-341,"-2,100","-4,157"
M4,269,"2,042",278,6,"1,293","3,172","7,060"
M5,-109,"-2,392",73,27,-153,"-3,679","-6,233"
M6,350,"3,029",148,50,"1,195","5,004","9,776"
M7,-155,"-3,484",48,52,-90,"-6,099","-9,728"
M8,184,"4,429",233,33,"1,300","8,463","14,642"
M9,-45,"-5,038",63,21,-199,"-10,517","-15,715"
M10,168,"6,149",196,31,"1,354","14,292","22,190"


In [46]:
# Iterate through each region to perform the inverse calculation
for region in calculated_well_counts_neg_allowed.keys():
    # Skip L48 as it's a total, not a region for individual GOR calculation
    if region == 'L48':
        continue

    # Get primary production for the current region
    primary_production = pPR_df[region].dropna().values

    # Check if the region column exists in npPR_df
    if region not in npPR_df.columns:
        print(f"Warning: No target non-primary production column found for region {region} in npPR.csv. Skipping.")
        calculated_gor_ratios[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target non-primary production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in npPR.csv
    target_non_primary_production = pd.to_numeric(npPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values #* 1000

    # Perform inverse GOR calculation
    monthly_gor_calculated = np.zeros(num_months)
    if region in GAS_REGIONS: # Primary is gas, non-primary is oil. Oil = Gas / GOR => GOR = Gas / Oil
        # Ensure target_non_primary_production is not zero to avoid division by zero
        monthly_gor_calculated = np.where(target_non_primary_production != 0, primary_production / target_non_primary_production, np.nan)

    else: # Primary is oil, non-primary is gas. Gas = Oil * GOR => GOR = Gas / Oil
        # Ensure primary_production is not zero to avoid division by zero
        monthly_gor_calculated = np.where(primary_production != 0, target_non_primary_production / primary_production, np.nan)

    calculated_gor_ratios[region] = monthly_gor_calculated

In [47]:
calculated_gor_ratios[region]

array([13.10997442, 12.88249634, 12.74215686, 12.68544831, 12.71813725,
       12.8007988 , 12.86529795, 12.64001986, 12.49260355, 12.30653021,
       12.21448999, 12.34600577, 12.66901408, 12.7011552 , 12.67223587,
       12.79653465, 12.6964462 , 12.6673287 , 12.9002006 , 12.86024218,
       12.85363409, 13.57876195, 13.32954545, 13.49871729])

In [48]:
# Create a DataFrame for the calculated GOR ratios
calculated_gor_ratios_df = pd.DataFrame(calculated_gor_ratios, index=month_cols)

print("\n--- Calculated GOR Ratios (Mcf/bbl or bbl/Mcf depending on primary) ---")
display(calculated_gor_ratios_df.style.format("{:,.2f}"))



--- Calculated GOR Ratios (Mcf/bbl or bbl/Mcf depending on primary) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48
M1,282.27,2.69,7.15,663.46,4.03,13.11
M2,284.02,2.66,7.01,628.00,4.02,12.88
M3,261.84,2.72,6.85,605.46,4.06,12.74
M4,253.77,2.75,6.37,593.44,4.06,12.69
M5,249.71,2.88,6.53,598.25,4.04,12.72
M6,249.81,2.90,6.33,622.70,4.14,12.80
M7,258.88,2.92,6.44,603.92,4.19,12.87
M8,245.51,2.93,6.24,630.78,4.19,12.64
M9,230.34,2.90,6.16,592.50,4.21,12.49
M10,236.65,2.83,6.32,562.20,4.24,12.31
